In [1]:
from os.path import basename, abspath
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
problem_name = basename(abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 00:53:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
products = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/products.txt")

sales = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/sales.txt")

In [5]:
products.createOrReplaceTempView("products")
sales.createOrReplaceTempView("sales")

In [6]:
products.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         1|      Laptop|
|         2|  Smartphone|
|         3|  Headphones|
|         4|       Mouse|
|         5|    Keyboard|
+----------+------------+



In [7]:
sales.show()

+-------+----------+---+
|sale_id|product_id|qty|
+-------+----------+---+
|   1001|         1| 10|
|   1002|         3|  5|
|   1003|         2|  8|
+-------+----------+---+



In [8]:
result = products.join(sales, on = (products.product_id == sales.product_id), how = "left")\
    .filter(sales.sale_id.isNull())\
    .select(products.product_id, products.product_name)

In [9]:
result.show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         4|       Mouse|
|         5|    Keyboard|
+----------+------------+



In [10]:
spark.sql(
    """
    select p.product_id, p.product_name
    from products p left join sales s on p.product_id = s.product_id
    where sale_id is null
    """
).show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|         4|       Mouse|
|         5|    Keyboard|
+----------+------------+



In [11]:
spark.stop()